# Notebook 9: Data Transformation


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PowerTransformer, MinMaxScaler, StandardScaler

values = pd.Series([10, 12, 14, 16, 20, 25, 30, 45, 70, 180], name="Sales")
print(values.to_list())

[10, 12, 14, 16, 20, 25, 30, 45, 70, 180]


## 1. Why Transform Data?

### Definition
Data transformation changes the representation or distribution of a feature. It can make patterns easier to learn and compare. Common transformations address skewness, different scales, or unusual ranges. Transformation does not automatically improve every dataset.

### Business Example
A shop has mostly small sales and a few very large orders. The largest orders dominate a chart of raw sales. Transforming sales can make smaller differences easier to see. The original amounts should still be retained for business reporting.

### AI/ML Use
Some models are sensitive to feature scale or extreme skewness. A suitable transformation can stabilize variation and improve optimization. Tree models often need less scaling. Fit learned transformations on training data only.

### Python Example
A quick log1p transform compresses large positive sales while keeping zero safe.

In [3]:
print(pd.DataFrame({"Raw": values, "Log1p": np.log1p(values)}).head())

   Raw     Log1p
0   10  2.397895
1   12  2.564949
2   14  2.708050
3   16  2.833213
4   20  3.044522


### Explanation
The code creates a new transformed column rather than overwriting the original. Large values shrink more than small values. The transformation changes the scale, not the order of sales. Check model performance before deciding to use it.

### When to Use / When Not to Use
Use when a feature has an unsuitable range or skew. Avoid changing meaningful units without recording the transformation.

## 2. Log Transformation

### Definition
A log transformation compresses high values more strongly than low values. It often reduces right skew in positive measurements. `np.log1p(x)` calculates log(1+x) and supports zero. Ordinary logarithms cannot handle zero or negative values directly.

### Business Example
Customer spending may range from ₹100 to ₹1,00,000. A few expensive purchases make most purchases hard to compare. Log spending gives a clearer view of typical customers. Actual rupee values remain important for reports.

### AI/ML Use
Log transforms are common for right-skewed income and transaction features. They can reduce the effect of large observations in linear models. The transformed relationship may be easier to model. They are not suitable for every feature.

### Python Example
Apply `np.log1p` to nonnegative sample sales.

In [5]:
print(np.log1p(values).round(2).to_list())

[2.4, 2.56, 2.71, 2.83, 3.04, 3.26, 3.43, 3.83, 4.26, 5.2]


### Explanation
The output shows smaller values on a compressed scale. Large sales no longer dominate by the same numerical distance. `log1p` works when a sale equals zero. Negative inputs require a different approach.

### When to Use / When Not to Use
Use for nonnegative right-skewed features. Do not use blindly on negative values or already symmetric data.

## 3. Square Root Transformation

### Definition
A square root transformation reduces the spread of large nonnegative values. It is gentler than a logarithm. It can help with moderately right-skewed counts. Square roots of negative real numbers are not defined.

### Business Example
A delivery company counts orders per hour. Most hours have few orders, while busy hours have many. Square root counts can reduce the imbalance in spread. The original order counts remain interpretable.

### AI/ML Use
Count features may benefit from square root transformation. It can make moderate skewness less severe. It does not guarantee a normal distribution. Compare distributions and model scores after applying it.

### Python Example
Use `np.sqrt` on nonnegative sales.

In [7]:
print(np.sqrt(values).round(2).to_list())

[3.16, 3.46, 3.74, 4.0, 4.47, 5.0, 5.48, 6.71, 8.37, 13.42]


### Explanation
Each transformed value is the square root of its original value. Larger values become closer together. The order of observations remains the same. The result is still nonnegative.

### When to Use / When Not to Use
Use for moderate right skew or count data. Do not use directly with negative observations.

## 4. Power Transformation

### Definition
Power transformations are a family of mathematical functions that reshape a distribution. They can reduce skewness and make variance more stable. Box-Cox and Yeo-Johnson are common choices. Their fitted parameters depend on the training data.

### Business Example
A retailer analyzes purchase amounts from different stores. Some stores have extremely skewed transactions. A power transformation may make the distributions easier to compare. The business should still retain the raw amounts.

### AI/ML Use
Power transformations can help linear and distance-based models. The fitted lambda parameter controls the change. A transformed feature is not guaranteed to become normal. Apply the same fitted transform to validation and test data.

### Python Example
Fit `PowerTransformer` to a small positive feature.

In [9]:
pt = PowerTransformer(method="yeo-johnson")
print(pt.fit_transform(values.to_frame()).ravel().round(2))

[-1.52 -1.13 -0.83 -0.58 -0.21  0.12  0.36  0.82  1.2   1.75]


### Explanation
The transformer estimates a parameter from the sample. Its default standardization also changes the scale. We print a few transformed values for inspection. Training-only fitting prevents leakage.

### When to Use / When Not to Use
Use for skewed numeric variables when validation supports it. Avoid unnecessary transformation of already suitable data.

## 5. Box-Cox Transformation

### Definition
Box-Cox is a power transformation for strictly positive values. It estimates a lambda parameter to reduce skewness. It cannot directly accept zero or negative numbers. The fitted transformation must be reused consistently.

### Business Example
A business studies positive monthly revenues. Revenue has a long right tail. Box-Cox may make the shape less skewed. Zero-revenue months need special consideration.

### AI/ML Use
Box-Cox can improve feature behavior for some statistical and ML models. It is especially relevant to positive continuous measurements. It must be fitted on training data. It is not always beneficial for tree-based methods.

### Python Example
Apply Box-Cox to strictly positive sales using `PowerTransformer`.

In [11]:
bc = PowerTransformer(method="box-cox")
print("Lambda:", bc.fit(values.to_frame()).lambdas_.round(2))
print(bc.transform(values.to_frame()).ravel()[:5].round(2))

Lambda: [-0.58]
[-1.54 -1.13 -0.82 -0.57 -0.2 ]


### Explanation
All input sales are greater than zero, so Box-Cox is valid. The transformer estimates lambda from the data. Output values are standardized by default. We print lambda to show the learned setting.

### When to Use / When Not to Use
Use for strictly positive skewed data. Do not apply directly to zeros or negatives.

## 6. Yeo-Johnson Transformation

### Definition
Yeo-Johnson is a power transformation that supports negative, zero, and positive values. Like Box-Cox, it estimates a lambda parameter. It can reduce skewness without manually shifting all values positive. It still may not produce a normal distribution.

### Business Example
Profit and loss figures can be negative, zero, or positive. A logarithm is not directly suitable for all these values. Yeo-Johnson can transform the entire series. Original profit figures remain necessary for interpretation.

### AI/ML Use
Yeo-Johnson can be useful for skewed features with mixed signs. It can help some models handle distributions more effectively. Fit it on training data only. Evaluate whether the change actually helps.

### Python Example
Apply Yeo-Johnson to profit values with mixed signs.

In [13]:
profits = pd.DataFrame({"Profit": [-100, -20, 0, 10, 40, 200]})
yj = PowerTransformer(method="yeo-johnson")
print(yj.fit_transform(profits).ravel().round(2))

[-1.66 -0.36 -0.08  0.03  0.33  1.73]


### Explanation
The example contains losses, zero, and gains. Yeo-Johnson accepts them without manual shifting. Lambda is learned from the sample. The printed values show the transformed feature.

### When to Use / When Not to Use
Use for skewed numeric data with zero or negative values. Avoid assuming it always improves prediction.

## 7. Handling Skewed Data

### Definition
Skewness measures asymmetry in a distribution. Positive skew has a longer right tail, while negative skew has a longer left tail. Transformations can reduce some types of skewness. Skewness alone does not prove that data is incorrect.

### Business Example
Online orders may be mostly small with a few very expensive purchases. That produces positive skew. A log transform can reduce the influence of the largest orders. Removing valid expensive purchases may lose important information.

### AI/ML Use
Some algorithms work better with less-skewed inputs. Others, including many tree models, are relatively robust. Compare feature distributions before and after transforming. Use validation performance to select the method.

### Python Example
Calculate skewness before and after a log transformation.

In [15]:
print("Before:", round(values.skew(), 2))
print("After:", round(np.log1p(values).skew(), 2))

Before: 2.52
After: 1.13


### Explanation
Pandas `skew()` summarizes asymmetry. We compare the original and log-transformed series. A lower absolute skewness can indicate a more symmetric distribution. It does not automatically mean the model is better.

### When to Use / When Not to Use
Use transformations when skew affects analysis or modeling. Do not remove legitimate extreme values solely to improve symmetry.

## 8. Normalization

### Definition
Normalization often means rescaling values to a fixed interval, commonly 0 to 1. Min-max scaling is one form of normalization. It preserves relative ordering but changes units. Extreme values can compress most observations into a narrow range.

### Business Example
A company compares customer age and spending on one chart. Their numerical ranges differ greatly. Normalization places both in a common range. Business users still need the original units for decisions.

### AI/ML Use
Distance-based algorithms can benefit from normalized features. Neural networks may also benefit from consistent scales. Fit the scaler only on training data. Outliers can strongly affect min-max results.

### Python Example
Normalize sales using `MinMaxScaler`.

In [17]:
scaler = MinMaxScaler()
print(scaler.fit_transform(values.to_frame()).ravel().round(2))

[0.   0.01 0.02 0.04 0.06 0.09 0.12 0.21 0.35 1.  ]


### Explanation
The smallest sample value becomes 0 and the largest becomes 1. Other values fall between them for this fitted dataset. Extreme sales influence where most values land. New test values may fall outside 0 to 1.

### When to Use / When Not to Use
Use for scale-sensitive methods needing bounded features. Avoid assuming min-max scaling is resistant to outliers.

## 9. Standardization

### Definition
Standardization subtracts the mean and divides by the standard deviation. It centers training values around zero with unit variance. It does not force the distribution to be normal. Outliers can affect the mean and standard deviation.

### Business Example
Employee salaries and years of experience use very different units. Standardization makes their scales more comparable. A salary z-score describes distance from average in standard-deviation units. Original values remain needed for business reports.

### AI/ML Use
Standardization often helps logistic regression, SVM, PCA, and KNN. It can make numerical optimization more stable. Fit the scaler on training data only. Outlier-heavy data may need a robust alternative.

### Python Example
Standardize sales using `StandardScaler`.

In [19]:
scaler = StandardScaler()
print(scaler.fit_transform(values.to_frame()).ravel().round(2))

[-0.66 -0.61 -0.57 -0.53 -0.45 -0.35 -0.25  0.06  0.57  2.8 ]


### Explanation
The transformed values have mean approximately zero for the fitted sample. Their population standard deviation is approximately one. Extreme values still influence the fitted statistics. This scaling is not the same as removing skewness.

### When to Use / When Not to Use
Use for scale-sensitive algorithms. Consider robust scaling when outliers dominate.